In [ ]:
#!pip install interpret

In [ ]:
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
set_visualize_provider(InlineProvider())

## The "Adult" dataset: predicting income

It contains about 30,000 observations and 14 socio-demographic and job related features from the 1994 Census database. The prediction task is determining if a person earns more than $50,000 annually.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    header=None)
df.columns = [
    "Age", "WorkClass", "fnlwgt", "Education", "EducationNum",
    "MaritalStatus", "Occupation", "Relationship", "Race", "Gender",
    "CapitalGain", "CapitalLoss", "HoursPerWeek", "NativeCountry", "Income"
]
X = df.iloc[:, :-1]
y = (df.iloc[:, -1] == " >50K").astype(int)

seed = 42
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=seed)

<h2>Explore the dataset</h2>

Most people in the dataset each below 50K anually.

In [ ]:
from interpret import show
from interpret.data import ClassHistogram

hist = ClassHistogram().explain_data(X_train, y_train, name='Train Data')
show(hist)

<h2>Train the Explainable Boosting Machine (EBM)</h2>

In general, EBM training can be very slow.

In [ ]:
from interpret.glassbox import ExplainableBoostingClassifier

ebm = ExplainableBoostingClassifier()
ebm.fit(X_train, y_train)

## Evaluate

- There is not much point in interpreting a model that is not reasonably accurate.
- Compare to a few black and glass boxes.

In [ ]:
from sklearn.metrics import roc_auc_score
print(f"EBM AUC: {roc_auc_score(y_test, ebm.predict_proba(X_test)[:,1]):.2f}")
print(f"EBM train AUC: {roc_auc_score(y_train, ebm.predict_proba(X_train)[:,1]):.2f}")

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

# We have to transform categorical variables before we can use Decision Tree in sklearn
X_tf = pd.get_dummies(X, prefix_sep='.').astype(float)
X_train_tf, X_test_tf, y_train, y_test = train_test_split(X_tf, y, test_size=0.20, random_state=seed)

# Train
gb_clf = GradientBoostingClassifier().fit(X_train_tf, y_train)
rf_clf = RandomForestClassifier().fit(X_train_tf, y_train)

# Print AUC values
print(f"Gradient Boosting AUC: {roc_auc_score(y_test, gb_clf.predict_proba(X_test_tf)[:,1]):.2f}")
print(f"Random Forest AUC: {roc_auc_score(y_test, rf_clf.predict_proba(X_test_tf)[:,1]):.2f}")

from interpret.glassbox import LogisticRegression, ClassificationTree

lr = LogisticRegression(random_state=seed, penalty='l1', solver='liblinear').fit(X_train_tf, y_train)
tree = ClassificationTree().fit(X_train_tf, y_train)

# Print AUC values
print(f"Logistic Regression AUC: {roc_auc_score(y_test, lr.predict_proba(X_test_tf)[:,1]):.2f}")
print(f"Classification Tree AUC: {roc_auc_score(y_test, tree.predict_proba(X_test_tf)[:,1]):.2f}")

## Global Explanations

Let us have an initial look at the model.

- In 'Summary', the terms are sorted by Mean Absolute Score
- 'Summary' does not display the intercept
- We can see that the model includes interactions, such as that of Age and Hours Per Week
- 14 interactions added overall
- The y-axis scale is, by default, the same for all plots, but can be adjusted
- We can get the overall score of a term from the 'Summary' view

In [ ]:
ebm_global = ebm.explain_global(name='EBM')
show(ebm_global)

### Exercises / Questions:
- Does the Age shape function look reasonable?
- What is the shape of the Capital Gain distribution?
- What does shape function of Capital Gain look like?
- Does a Capital Gain of around 50K increase the chances of earning >50K?
- How does Age relate to Working Hours?
- How informative does Native Country seem to be? Autoscale to better see differences between countries.
- Is being divorced associated with higher chance of earning >50K?
- How do chances of earning >50K relate to Work Hours for those with Married to a civilian spouse?

<h2>Local Explanations: How an individual prediction was made</h2>

Let's look at the local explanations for the first 5 instances in the test set.


In [ ]:
X_test[:5]

- The terms are sorted by magnitude (absolute value).
- Unlike in the global summary, here we can visualize the intercept.
- For instance 0, only a few terms increase probability of >50K, but not nearly enough to compensate for the intercept and other terms.
- For instance 2, most terms increase probability of >50K, enough to compensate for the intercept and other terms.

In [ ]:
ebm_local = ebm.explain_local(X_test[:5], y_test[:5], name='EBM')
show(ebm_local, 0)

### Exercises / Questions
- If instance 1 were male, would the prediction change?


## Binning and shape functions somplexity

While the default maximum number of bins is 1042, EBM has used much less. For example, Age is binned in the following grups for its shape function:

In [ ]:
print(ebm.feature_names_in_[0])
ebm.bins_[0][0]
len(ebm.bins_[0][0])

And in the following groups for all interactions:

In [ ]:
ebm.bins_[0][1]

Question: why less bins for interactions?

What about Capital Gain? Indeed, its shape function rather wiggly.

In [ ]:
print(ebm.feature_names_in_[10])
len(ebm.bins_[10][0])

Let's see how well does EBM do with a train set of roughly 300 instances:

- How has the ROC AUC changed?
- How has the Capital Gain shape function changed?

Early stopping can help prevent overfitting.

In [ ]:
seed = 42
np.random.seed(seed)
X_train_small, X_test_small, y_train_small, y_test_small = train_test_split(X, y, test_size=0.99, random_state=seed)

In [ ]:
ebm_small = ExplainableBoostingClassifier().fit(X_train_small, y_train_small)

roc_auc_score(y_test_small, ebm_small.predict_proba(X_test_small)[:,1])

In [ ]:
ebm_global = ebm_small.explain_global(name='EBM')
show(ebm_global)

## Editing the model

- We may want to add constraints such as the shape function being monotonic, or otherwise edit a shape function given some external knowledge.
- For example, lets monotonize the shape functions for Age and Capital Gain. Before doing that, let's print the intercept.

In [ ]:
ebm.term_scores_[0].mean()

In [ ]:
print(f"Intercept: {ebm.intercept_[0]:.2f}")
ebm.monotonize("Age", increasing=True)
ebm.monotonize("CapitalGain", increasing=True)
ebm_global = ebm.explain_global(name='EBM')
show(ebm_global)

### Exercises / Questions
- Have the intercept and other shape functions changed? What is their mean value?
- How well is the model doing on training / test data?

- In addition to monotonizing, we can scale the contribution of a feature by a constant factor $a$. Note how its relevance is multiplied by $a$.

In [ ]:
ebm.scale("Gender", 2)
show(ebm.explain_global(name='EBM'))

### Exercises / Questions
- What are some advantages of modifying the model versus re-training it?
- Make term "WorkClass" irrelevant by scaling it by an appropriate factor.

## Exercise

Retrain EBM with the following arguments; then evaluate ROC and look at the principal shape functions.

- Limit number of interactions to 2 (`interactions = 2`).
- Limit the number of rounds to 100 (`max_rounds`).
- Set learning rate to 1 (`learning_rate`)
- Reduce validation set size (default `validation_size=0.15`)
- Try combining some of the above changes.

## Exercise: the Breast Cancer dataset

- Train an EBM.
- Evaluate the tree and compare to blackbox and glassbox models.
- What is the relevance of the "mean perimeter" used by the tree?
- Can you find some interesting patterns in the shape functions?
- Try to change parameters to improve performance.
